# Synthetic Data Generator
### Step 11 : Build Synthetic Final Aligned Output Stage

This notebook builds the final Kafka/Postgres synthetic output so it matches the Kaggle-style dataset shape:

- `dataset_id`
- `run_id`
- `asset_id`
- `timestamp`
- `sensor_00` ... `sensor_51`
- `machine_status`

It uses the rebuilt stage as the source table and does **not** read premelt for this final output step.


In [12]:
import os
import pandas as pd 

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
    execute_sql,
)

from utils.synthetic.pipeline.final_aligned_output_writer import build_synthetic_final_aligned_output_stage

from utils.synthetic.pipeline.final_aligned_incremental import run_final_align_loop

from utils.core.env_helpers import (
    env_bool, 
    env_int, 
    env_optional_int, 
    env_str,
)

pd.set_option("display.max_colwidth", None)



In [ ]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

NUMBER_OF_SENSORS = env_int("SYNTHETIC_SENSOR_COUNT", 52)

IF_EXISTS_FLAG = "replace"

COMPLETE_ONLY_FLAG = env_bool(
    "SYNTHETIC_FINAL_ALIGN_COMPLETE_ONLY",
    True,
)

OBSERVATION_WINDOW_SIZE = env_int(
    "SYNTHETIC_REBUILD_OBSERVATION_WINDOW_SIZE",
    2500,
)

BATCH_SIZE = env_int(
    "SYNTHETIC_FINAL_ALIGN_BATCH_SIZE",
    1000,
)

MAX_ITERATIONS = env_optional_int(
    "SYNTHETIC_FINAL_ALIGN_MAX_ITERATIONS",
    default=None,
)

STOP_ON_FAILURE = env_bool(
    "SYNTHETIC_FINAL_ALIGN_STOP_ON_FAILURE",
    True,
)

PREMELT_TABLE_NAME = env_str(
    "SYNTHETIC_PREMELT_OBSERVATIONS_TABLE",
    "synthetic_observations_premelt_stage",
)

REBUILT_TABLE_NAME = env_str(
    "SYNTHETIC_REBUILT_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_rebuilt_stage",
)

TARGET_TABLE_NAME = env_str(
    "SYNTHETIC_FINAL_ALIGNED_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_final_aligned_stage",
    aliases=("FINAL_ALIGNMENT_SOURCE_TABLE_NAME",),
)

TIMESTAMP_SOURCE_PRIORITY = (
    "observation_timestamp",
    "timestamp",
    "created_at",
    "event_time",
)

STATUS_SOURCE_PRIORITY = (
    "machine_status",
    "stream_state",
    "phase",
)

STATUS_MAPPING = {
    "normal": "NORMAL",
    "broken": "BROKEN",
    "abnormal": "BROKEN",
    "failure": "BROKEN",
    "failed": "BROKEN",
    "fault": "BROKEN",
    "recovering": "RECOVERING",
    "recovery": "RECOVERING",
}

print("Final aligned config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("premelt table:", PREMELT_TABLE_NAME)
print("rebuilt table:", REBUILT_TABLE_NAME)
print("target table:", TARGET_TABLE_NAME)
print("batch size:", BATCH_SIZE)
print("max iterations:", MAX_ITERATIONS)

Final aligned config
schema: capstone
dataset id: pump_synthetic_v1
run id: synthetic_run_001
premelt table: synthetic_observations_premelt_stage
rebuilt table: synthetic_sensor_observations_rebuilt_stage
target table: synthetic_sensor_observations_final_aligned_stage
batch size: 1000
max iterations: None


---

In [9]:
engine = get_engine_from_env()

---

In [ ]:
'''

# Full Run

final_output_result = build_synthetic_final_aligned_output_stage(
    engine=engine,
    schema=SCHEMA,
    rebuilt_table=REBUILT_TABLE_NAME,
    target_table=TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    if_exists=IF_EXISTS_FLAG,
    observation_window_size=OBSERVATION_WINDOW_SIZE,
    timestamp_source_priority=TIMESTAMP_SOURCE_PRIORITY,
    status_source_priority=STATUS_SOURCE_PRIORITY,
    status_mapping=STATUS_MAPPING,
    timestamp_output_column="timestamp",
    machine_status_output_column="machine_status",
)

display(final_output_result)

'''

In [10]:
final_aligned_columns_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        ordinal_position,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = :schema_name
      AND table_name = :table_name
    ORDER BY ordinal_position
    """,
    params={
        "schema_name": SCHEMA,
        "table_name": TARGET_TABLE_NAME,
    },
)

display(final_aligned_columns_dataframe)

,ordinal_position,column_name,data_type
0,1,aligned_observation_id,bigint
1,2,dataset_id,text
2,3,run_id,text
3,4,asset_id,text
4,5,event_time,timestamp with time zone
5,6,event_step,bigint
6,7,time_index,bigint
7,8,machine_status,text
8,9,operational_state,text
9,10,anomaly_flag,integer


In [ ]:
'''
execute_sql(
    engine,
    f"""
    DROP TABLE IF EXISTS "{SCHEMA}"."{TARGET_TABLE_NAME}" CASCADE
    """
)

print(f"Dropped stale Stage 11 target table: {SCHEMA}.{TARGET_TABLE_NAME}")
'''

Dropped stale Stage 11 target table: capstone.synthetic_sensor_observations_final_aligned_stage


In [ ]:
# Batched Incremental Run

final_align_results = run_final_align_loop(
    engine=engine,
    batch_size=BATCH_SIZE,
    schema=SCHEMA,
    premelt_table=PREMELT_TABLE_NAME,
    rebuilt_table=REBUILT_TABLE_NAME,
    target_table=TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    prefer_rebuilt_sensor_values=True,
    max_iterations=MAX_ITERATIONS,
    stop_on_failure=STOP_ON_FAILURE,
)

display(pd.DataFrame(final_align_results))

[final-align-incremental] Added 54 new columns to capstone.synthetic_sensor_observations_final_aligned_stage


----

In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT dataset_id) AS dataset_id_count,
        COUNT(DISTINCT run_id) AS run_id_count,
        COUNT(DISTINCT asset_id) AS asset_id_count,
        MIN(event_time) AS min_event_time,
        MAX(event_time) AS max_event_time,
        COUNT(*) FILTER (WHERE machine_status = 'NORMAL') AS normal_rows,
        COUNT(*) FILTER (WHERE machine_status = 'BROKEN') AS broken_rows,
        COUNT(*) FILTER (WHERE machine_status = 'RECOVERING') AS recovering_rows
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    """
)

display(validation_dataframe)

---


### Sample Output

This shows the final Kaggle-shaped output table layout.


In [ ]:
sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        event_time,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        machine_status
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    ORDER BY timestamp, dataset_id, run_id, asset_id
    LIMIT 10
    """
)

display(sample_dataframe)


---

In [ ]:
status_distribution_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        machine_status,
        COUNT(*) AS row_count
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    GROUP BY machine_status
    ORDER BY machine_status
    """
)

display(status_distribution_dataframe)


---

In [ ]:
stage_11_final_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS final_row_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(DISTINCT generated_row_id) AS distinct_generated_row_id_count,
        COUNT(*) FILTER (WHERE observation_index IS NULL) AS null_observation_index_count,
        COUNT(*) FILTER (WHERE generated_row_id IS NULL) AS null_generated_row_id_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS rebuild_complete_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        MIN(observation_timestamp) AS min_observation_timestamp,
        MAX(observation_timestamp) AS max_observation_timestamp,
        (
            COUNT(*) = 225000
            AND COUNT(DISTINCT observation_index) = 225000
            AND COUNT(DISTINCT generated_row_id) = 225000
            AND COUNT(*) FILTER (WHERE observation_index IS NULL) = 0
            AND COUNT(*) FILTER (WHERE generated_row_id IS NULL) = 0
            AND COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) = 225000
            AND MIN(observation_index) = 1
            AND MAX(observation_index) = 225000
        ) AS ready_for_final_output_export
    FROM "{SCHEMA}"."synthetic_sensor_observations_final_aligned_stage"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_11_final_validation_dataframe)

---

In [ ]:
# Dispose Engine
engine.dispose()